# Insurance Claim Processing Agent (LangGraph)
- **Model:** `gemini-3.5-flash-lite`
- **5 Nodes:** `document_verification_agent`, `eligibility_check_agent`, `fraud_detection_agent` (executed in parallel), `claim_summary_agent`, and `human_approval_agent`.

In [4]:
import os
from typing import TypedDict
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END

# Load single GOOGLE_API_KEY from .env
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite", google_api_key=GOOGLE_API_KEY, temperature=0
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


### Define Claim State, 5 Agents, Parallel Edges, and Conditional Routing

In [5]:
class ClaimState(TypedDict, total=False):
    claim_id: str
    policy_status: str
    claim_amount: float
    documents_complete: bool
    suspicious_activity: bool
    doc_check: str
    eligibility_check: str
    fraud_check: str
    claim_summary: str
    decision: str
    human_review: str


# Node 1: Document Verification Agent
def document_verification_agent(state: ClaimState):
    return {
        "doc_check": "Verified" if state["documents_complete"] else "Missing Documents"
    }


# Node 2: Eligibility Check Agent
def eligibility_check_agent(state: ClaimState):
    return {
        "eligibility_check": "Eligible"
        if state["policy_status"] == "Active"
        else "Expired Policy"
    }


# Node 3: Fraud Detection Agent
def fraud_detection_agent(state: ClaimState):
    if state["suspicious_activity"] or state["claim_amount"] > 25000:
        return {"fraud_check": "High Risk / High Value"}
    return {"fraud_check": "Low Risk"}


# Node 4: Claim Summary Agent (Extracts only plain text, no metadata)
def claim_summary_agent(state: ClaimState):
    if state["doc_check"] != "Verified":
        decision = "Reject"
    elif state["eligibility_check"] != "Eligible":
        decision = "Reject"
    elif state["fraud_check"] != "Low Risk":
        decision = "Human Approval"
    else:
        decision = "Auto Approve"

    prompt = (
        f"Summarize insurance claim {state['claim_id']} for ${state['claim_amount']} in 1 sentence. "
        f"Docs: {state['doc_check']}, Policy: {state['eligibility_check']}, "
        f"Fraud: {state['fraud_check']}, Decision: {decision}."
    )
    content = llm.invoke(prompt).content
    if isinstance(content, list):
        summary = " ".join(
            block.get("text", "") if isinstance(block, dict) else str(block)
            for block in content
        ).strip()
    else:
        summary = str(content).strip()

    return {"claim_summary": summary, "decision": decision}


# Node 5: Human Approval Agent
def human_approval_agent(state: ClaimState):
    return {"human_review": "Escalated to Human Reviewer for manual approval."}


# Conditional Router
def route_claim(state: ClaimState):
    if state["decision"] == "Human Approval":
        return "human_approval_agent"
    return END


# Build LangGraph
builder = StateGraph(ClaimState)
builder.add_node("document_verification_agent", document_verification_agent)
builder.add_node("eligibility_check_agent", eligibility_check_agent)
builder.add_node("fraud_detection_agent", fraud_detection_agent)
builder.add_node("claim_summary_agent", claim_summary_agent)
builder.add_node("human_approval_agent", human_approval_agent)

# Parallel execution of verification, eligibility, and fraud checks
builder.add_edge(START, "document_verification_agent")
builder.add_edge(START, "eligibility_check_agent")
builder.add_edge(START, "fraud_detection_agent")

builder.add_edge("document_verification_agent", "claim_summary_agent")
builder.add_edge("eligibility_check_agent", "claim_summary_agent")
builder.add_edge("fraud_detection_agent", "claim_summary_agent")

builder.add_conditional_edges("claim_summary_agent", route_claim)
builder.add_edge("human_approval_agent", END)

claim_graph = builder.compile()

### Run All 5 Claim Scenarios (Input -> Output)

In [6]:
test_cases = [
    {
        "claim_id": "CLM-101",
        "policy_status": "Active",
        "claim_amount": 4000,
        "documents_complete": True,
        "suspicious_activity": False,
    },
    {
        "claim_id": "CLM-102",
        "policy_status": "Active",
        "claim_amount": 5000,
        "documents_complete": False,
        "suspicious_activity": False,
    },
    {
        "claim_id": "CLM-103",
        "policy_status": "Active",
        "claim_amount": 15000,
        "documents_complete": True,
        "suspicious_activity": True,
    },
    {
        "claim_id": "CLM-104",
        "policy_status": "Expired",
        "claim_amount": 3000,
        "documents_complete": True,
        "suspicious_activity": False,
    },
    {
        "claim_id": "CLM-105",
        "policy_status": "Active",
        "claim_amount": 35000,
        "documents_complete": True,
        "suspicious_activity": False,
    },
]

for tc in test_cases:
    result = claim_graph.invoke(tc)
    print(f"Claim {tc['claim_id']} -> Decision: {result['decision']}")
    print(f"Summary: {result['claim_summary']}")
    if "human_review" in result:
        print(f"HITL: {result['human_review']}")
    print("-" * 60)

c:\Users\singh\OneDrive\Desktop\assignment gen ai\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Claim CLM-101 -> Decision: Auto Approve
Summary: Insurance claim CLM-101 for $4,000 has been automatically approved following document verification, policy eligibility confirmation, and a low-risk fraud assessment.
------------------------------------------------------------


c:\Users\singh\OneDrive\Desktop\assignment gen ai\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Claim CLM-102 -> Decision: Reject
Summary: Insurance claim CLM-102 for $5,000 has been rejected despite being policy-eligible and low risk for fraud, due to missing documents.
------------------------------------------------------------


c:\Users\singh\OneDrive\Desktop\assignment gen ai\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Claim CLM-103 -> Decision: Human Approval
Summary: Insurance claim CLM-103 for $15,000 has verified documents and an eligible policy, but due to its high-risk and high-value fraud status, it has been routed for human approval.
HITL: Escalated to Human Reviewer for manual approval.
------------------------------------------------------------


c:\Users\singh\OneDrive\Desktop\assignment gen ai\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Claim CLM-104 -> Decision: Reject
Summary: Insurance claim CLM-104 for $3,000 has been rejected due to an expired policy, despite being verified and classified as low risk for fraud.
------------------------------------------------------------


c:\Users\singh\OneDrive\Desktop\assignment gen ai\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Claim CLM-105 -> Decision: Human Approval
Summary: Insurance claim CLM-105 for $35,000 has verified documents and an eligible policy, but due to being a high-risk and high-value fraud case, it has been routed for human approval.
HITL: Escalated to Human Reviewer for manual approval.
------------------------------------------------------------
